# ETL — Global Video Game Sales
**Projeto Final de BI**

Fonte: [Kaggle — thedevastator/global-video-game-sales](https://www.kaggle.com/datasets/thedevastator/global-video-game-sales)

---
## Ferramentas utilizadas

| Componente | Tecnologia | Aplicação |
|---|---|---|
| **Banco de dados** | PostgreSQL 15 (Docker) | Data Warehouse em Star Schema |
| **Interface BI** | Plotly Dash | Dashboard web interativo (`dashboard_app.py`) |
| **ETL** | Python 3 · Pandas · SQLAlchemy · psycopg2 | Extract → Transform → Load |
| **Coleta de dados** | CSV local | Arquivo `vgsales.csv` |

---
## Fluxo do projeto

```
┌──────────────────────────────────────────────────────┐
│  etl_videogames.ipynb  (este arquivo)                │
│                                                      │
│  [0] Instalar deps → [1] Configuração                │
│  [2] Conexão PG   → [3] Extract CSV                  │
│  [4] Transform    → [5] Load                         │
│  [6] Validar                                         │
└──────────────────────────────────────────────────────┘
```

## 0. Dependências

In [ ]:
# !pip install -q pandas sqlalchemy psycopg2-binary

## 1. Configuração

Ajuste `CSV_PATH` e as credenciais do banco conforme necessário.

In [ ]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

CSV_PATH = 'vgsales.csv'

if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(
        f'CSV não encontrado em "{CSV_PATH}".\n'
        'Baixe o dataset em: https://www.kaggle.com/datasets/thedevastator/global-video-game-sales\n'
        'e coloque o vgsales.csv na mesma pasta deste notebook.'
    )

print(f'CSV encontrado: {CSV_PATH}  ({os.path.getsize(CSV_PATH):,} bytes)')

# Banco de dados
DB_USER     = 'postgres'
DB_PASSWORD = 'postgres'
DB_HOST     = 'localhost'
DB_PORT     = '5433'
DB_NAME     = 'dw_projeto_final'

DB_URL = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

CSV encontrado: vgsales.csv  (1,355,781 bytes)


## 2. Conexão com PostgreSQL

In [2]:
engine = create_engine(DB_URL, echo=False)

with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
print('Conexão com PostgreSQL OK.')

Conexão com PostgreSQL OK.


## 3. EXTRACT — Leitura do CSV

In [3]:
df_raw = pd.read_csv(CSV_PATH)

print(f'Shape bruto: {df_raw.shape}')
print('\nValores nulos por coluna:')
print(df_raw.isnull().sum())
df_raw.head(5)

Shape bruto: (16598, 11)

Valores nulos por coluna:
Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


## 4. TRANSFORM — Limpeza e modelagem Star Schema

### Tabelas geradas:
- **`fato_vendas`** — métricas de vendas (NA, EU, JP, Other, Global)
- **`dim_jogo`** — títulos únicos
- **`dim_plataforma`** — consoles enriquecidos com fabricante e geração
- **`dim_genero`** — gêneros
- **`dim_publisher`** — publicadoras
- **`dim_ano`** — anos com era da indústria calculada

In [4]:
df = df_raw.copy()
df.columns = ['rank', 'name', 'platform', 'year', 'genre', 'publisher',
               'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'global_sales']

# Limpeza básica
df = df.dropna(subset=['name', 'platform'])
df['year']      = pd.to_numeric(df['year'], errors='coerce')
df['publisher'] = df['publisher'].fillna('Unknown')
df['genre']     = df['genre'].fillna('Unknown')
for col in ['na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'global_sales']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0)

print(f'Registros após limpeza: {len(df):,}')

Registros após limpeza: 16,598


In [ ]:
# dim_plataforma
geracao_map = {
    'NES': '5ª Geração',  'SNES': '5ª Geração', 'GB':   '5ª Geração',
    'GEN': '5ª Geração',  '2600': '4ª Geração', 'SCD':  '5ª Geração',
    'N64': '6ª Geração',  'PS':   '6ª Geração', 'SAT':  '6ª Geração',
    'GBA': '6ª Geração',  'DC':   '6ª Geração', 'PS2':  '7ª Geração',
    'GC':  '7ª Geração',  'XB':   '7ª Geração', 'DS':   '7ª Geração',
    'Wii': '8ª Geração',  'X360': '8ª Geração', 'PS3':  '8ª Geração',
    'PSP': '8ª Geração',  '3DS':  '8ª Geração', 'PS4':  '9ª Geração',
    'XOne':'9ª Geração',  'WiiU': '9ª Geração', 'PSV':  '9ª Geração',
    'PC':  'PC',
}
fabricante_map = {
    'NES': 'Nintendo', 'SNES': 'Nintendo', 'N64': 'Nintendo', 'GC':  'Nintendo',
    'Wii': 'Nintendo', 'WiiU': 'Nintendo', 'GB':  'Nintendo', 'GBA': 'Nintendo',
    'DS':  'Nintendo', '3DS':  'Nintendo',
    'PS':  'Sony',     'PS2':  'Sony',     'PS3': 'Sony',     'PS4': 'Sony',
    'PSP': 'Sony',     'PSV':  'Sony',
    'XB':  'Microsoft','X360': 'Microsoft','XOne':'Microsoft',
    'DC':  'Sega',     'GEN':  'Sega',     'SAT': 'Sega',     'SCD': 'Sega',
    'PC':  'PC',       '2600': 'Atari',
}

dim_plataforma = (df[['platform']].drop_duplicates()
                  .reset_index(drop=True)
                  .rename(columns={'platform': 'nome_plataforma'}))
dim_plataforma['id_plataforma'] = dim_plataforma.index + 1
dim_plataforma['fabricante']    = dim_plataforma['nome_plataforma'].map(fabricante_map).fillna('Outro')
dim_plataforma['geracao']       = dim_plataforma['nome_plataforma'].map(geracao_map).fillna('Outro')

# dim_genero
dim_genero = (df[['genre']].drop_duplicates().reset_index(drop=True)
              .rename(columns={'genre': 'nome_genero'}))
dim_genero['id_genero'] = dim_genero.index + 1

# dim_publisher
dim_publisher = (df[['publisher']].drop_duplicates().reset_index(drop=True)
                 .rename(columns={'publisher': 'nome_publisher'}))
dim_publisher['id_publisher'] = dim_publisher.index + 1

# dim_jogo
dim_jogo = (df[['name']].drop_duplicates().reset_index(drop=True)
            .rename(columns={'name': 'nome_jogo'}))
dim_jogo['id_jogo'] = dim_jogo.index + 1

# dim_ano
def classifica_era(ano):
    if ano < 1985: return 'Era dos Arcades'
    if ano < 1995: return 'Era 16-bit'
    if ano < 2000: return 'Era 32/64-bit'
    if ano < 2006: return 'Era 6ª Geração'
    if ano < 2013: return 'Era HD'
    return 'Era Atual'

anos_validos = sorted(df['year'].dropna().astype(int).unique())
dim_ano = pd.DataFrame({'ano': anos_validos})
dim_ano['id_ano']        = dim_ano.index + 1
dim_ano['era_industria'] = dim_ano['ano'].apply(classifica_era)

print('Dimensões construídas:')
for nome, d in [('dim_plataforma', dim_plataforma), ('dim_genero', dim_genero),
                ('dim_publisher',  dim_publisher),  ('dim_jogo',   dim_jogo),
                ('dim_ano',        dim_ano)]:
    print(f'  {nome}: {len(d)} registros')

Dimensões construídas:
  dim_plataforma: 31 registros
  dim_genero: 12 registros
  dim_publisher: 578 registros
  dim_jogo: 11493 registros
  dim_ano: 39 registros


In [ ]:
# fato_vendas
fato = df.copy()
fato = fato.merge(dim_jogo.rename(columns={'nome_jogo': 'name'}),                on='name')
fato = fato.merge(dim_plataforma.rename(columns={'nome_plataforma': 'platform'}), on='platform')
fato = fato.merge(dim_genero.rename(columns={'nome_genero': 'genre'}),            on='genre')
fato = fato.merge(dim_publisher.rename(columns={'nome_publisher': 'publisher'}),  on='publisher')
fato = fato.merge(dim_ano[['ano', 'id_ano']], left_on='year', right_on='ano', how='left')

fato_final = (fato[['id_jogo', 'id_plataforma', 'id_genero', 'id_publisher', 'id_ano',
                     'rank', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'global_sales']]
              .copy()
              .rename(columns={'rank': 'rank_global'}))
fato_final['id_ano'] = fato_final['id_ano'].astype('Int64')

print(f'fato_vendas: {len(fato_final):,} registros')
fato_final.head()

fato_vendas: 16,598 registros


,id_jogo,id_plataforma,id_genero,id_publisher,id_ano,rank_global,na_sales,eu_sales,jp_sales,other_sales,global_sales
0,1,1,1,1,27,1,41.49,29.02,3.77,8.46,82.74
1,2,2,2,1,6,2,29.08,3.58,6.81,0.77,40.24
2,3,1,3,1,29,3,15.85,12.88,3.79,3.31,35.82
3,4,1,1,1,30,4,15.75,11.01,3.28,2.96,33.00
4,5,3,4,1,17,5,11.27,8.89,10.22,1.00,31.37


## 5. LOAD — Carga no PostgreSQL

In [ ]:
ddl = """
DROP TABLE IF EXISTS fato_vendas    CASCADE;
DROP TABLE IF EXISTS dim_jogo       CASCADE;
DROP TABLE IF EXISTS dim_plataforma CASCADE;
DROP TABLE IF EXISTS dim_genero     CASCADE;
DROP TABLE IF EXISTS dim_publisher  CASCADE;
DROP TABLE IF EXISTS dim_ano        CASCADE;

CREATE TABLE dim_plataforma (
    id_plataforma   SERIAL PRIMARY KEY,
    nome_plataforma VARCHAR(50)  NOT NULL,
    fabricante      VARCHAR(50),
    geracao         VARCHAR(30)
);
CREATE TABLE dim_genero (
    id_genero   SERIAL PRIMARY KEY,
    nome_genero VARCHAR(50) NOT NULL
);
CREATE TABLE dim_publisher (
    id_publisher   SERIAL PRIMARY KEY,
    nome_publisher VARCHAR(200) NOT NULL
);
CREATE TABLE dim_ano (
    id_ano        SERIAL PRIMARY KEY,
    ano           INTEGER NOT NULL,
    era_industria VARCHAR(30)
);
CREATE TABLE dim_jogo (
    id_jogo   SERIAL PRIMARY KEY,
    nome_jogo VARCHAR(300) NOT NULL
);
CREATE TABLE fato_vendas (
    id_fato       SERIAL PRIMARY KEY,
    id_jogo       INTEGER REFERENCES dim_jogo(id_jogo),
    id_plataforma INTEGER REFERENCES dim_plataforma(id_plataforma),
    id_genero     INTEGER REFERENCES dim_genero(id_genero),
    id_publisher  INTEGER REFERENCES dim_publisher(id_publisher),
    id_ano        INTEGER REFERENCES dim_ano(id_ano),
    rank_global   INTEGER,
    na_sales      NUMERIC(8,2),
    eu_sales      NUMERIC(8,2),
    jp_sales      NUMERIC(8,2),
    other_sales   NUMERIC(8,2),
    global_sales  NUMERIC(8,2)
);
CREATE INDEX idx_fv_jogo      ON fato_vendas(id_jogo);
CREATE INDEX idx_fv_plat      ON fato_vendas(id_plataforma);
CREATE INDEX idx_fv_genero    ON fato_vendas(id_genero);
CREATE INDEX idx_fv_publisher ON fato_vendas(id_publisher);
CREATE INDEX idx_fv_ano       ON fato_vendas(id_ano);
"""

with engine.connect() as conn:
    conn.execute(text(ddl))
    conn.commit()
print('Schema recriado com sucesso.')

In [ ]:
dim_plataforma[['nome_plataforma', 'fabricante', 'geracao']].to_sql('dim_plataforma', engine, if_exists='append', index=False)
dim_genero[['nome_genero']].to_sql('dim_genero',           engine, if_exists='append', index=False)
dim_publisher[['nome_publisher']].to_sql('dim_publisher',  engine, if_exists='append', index=False)
dim_ano[['ano', 'era_industria']].to_sql('dim_ano',        engine, if_exists='append', index=False)
dim_jogo[['nome_jogo']].to_sql('dim_jogo',                 engine, if_exists='append', index=False)
print('Dimensões carregadas.')

fato_final.to_sql('fato_vendas', engine, if_exists='append', index=False, chunksize=500)
print(f'fato_vendas carregada: {len(fato_final):,} linhas.')

## 6. Validação — Queries de Negócio

In [ ]:
queries = {
    'Top 10 jogos (global)': """
        SELECT j.nome_jogo, p.nome_plataforma,
               ROUND(SUM(f.global_sales)::numeric,2) AS total_mm
        FROM fato_vendas f
        JOIN dim_jogo j       ON f.id_jogo = j.id_jogo
        JOIN dim_plataforma p ON f.id_plataforma = p.id_plataforma
        GROUP BY j.nome_jogo, p.nome_plataforma
        ORDER BY total_mm DESC LIMIT 10;
    """,
    'Vendas por gênero': """
        SELECT g.nome_genero,
               ROUND(SUM(f.global_sales)::numeric,2) AS global,
               ROUND(SUM(f.na_sales)::numeric,2)     AS na,
               ROUND(SUM(f.eu_sales)::numeric,2)     AS eu,
               ROUND(SUM(f.jp_sales)::numeric,2)     AS jp
        FROM fato_vendas f
        JOIN dim_genero g ON f.id_genero = g.id_genero
        GROUP BY g.nome_genero ORDER BY global DESC;
    """,
    'Top 10 publishers': """
        SELECT pub.nome_publisher,
               COUNT(DISTINCT f.id_jogo)             AS qtd_jogos,
               ROUND(SUM(f.global_sales)::numeric,2) AS total_mm
        FROM fato_vendas f
        JOIN dim_publisher pub ON f.id_publisher = pub.id_publisher
        WHERE pub.nome_publisher != 'Unknown'
        GROUP BY pub.nome_publisher ORDER BY total_mm DESC LIMIT 10;
    """,
    'Evolução anual': """
        SELECT a.ano,
               COUNT(f.id_fato)                        AS lancamentos,
               ROUND(SUM(f.global_sales)::numeric,2)   AS total_mm
        FROM fato_vendas f
        JOIN dim_ano a ON f.id_ano = a.id_ano
        WHERE a.ano BETWEEN 1980 AND 2016
        GROUP BY a.ano ORDER BY a.ano;
    """,
    'Share por fabricante': """
        SELECT p.fabricante,
               ROUND(SUM(f.na_sales)::numeric,2)     AS na,
               ROUND(SUM(f.eu_sales)::numeric,2)     AS eu,
               ROUND(SUM(f.jp_sales)::numeric,2)     AS jp,
               ROUND(SUM(f.global_sales)::numeric,2) AS global
        FROM fato_vendas f
        JOIN dim_plataforma p ON f.id_plataforma = p.id_plataforma
        GROUP BY p.fabricante ORDER BY global DESC;
    """,
    'Era da indústria': """
        SELECT a.era_industria,
               COUNT(f.id_fato)                        AS lancamentos,
               ROUND(SUM(f.global_sales)::numeric,2)   AS total_mm,
               ROUND(AVG(f.global_sales)::numeric,4)   AS media_por_jogo
        FROM fato_vendas f
        JOIN dim_ano a ON f.id_ano = a.id_ano
        GROUP BY a.era_industria ORDER BY total_mm DESC;
    """,
}

with engine.connect() as conn:
    for titulo, sql in queries.items():
        print(f'\n── {titulo} ──')
        display(pd.read_sql(text(sql), conn))